In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd



# Basic debugging functions
def print_tensor_details(tensor):
    print('instance before debug: ', tensor)
    print('shape: ', tensor.shape)
    print('size(): ', tensor.size())
    print('device: ', tensor.device)
    print('dtype: ', tensor.dtype)
    print('grad: ', tensor.grad)
    print('grad_fn: ', tensor.grad_fn)
    # print('Tensor Row/Col', tensor[:, :])
    # print('Tensor Col-2nd', tensor[:, 1])
    # print('Tensor Row-2nd', tensor[1, :])
    print('instance after debug: ', tensor)
    pass


In [2]:
# Sample Data - List of lists (2D Matrix)
data = [[1.0, 2.0, 3.0, 4.0],
        [10.0, 20.0, 30.0, 40.0],
        [100.0, 200.0, 300.0, 400.0]]


row_vect = [[2000, 3000, 5000]]
print("Row vect: ", row_vect, " SHape: ", np.shape(row_vect))

col_vect = np.reshape(row_vect, (3, 1))
print("Col vect: \n", col_vect, " SHape: ", np.shape(col_vect))

Row vect:  [[2000, 3000, 5000]]  SHape:  (1, 3)
Col vect: 
 [[2000]
 [3000]
 [5000]]  SHape:  (3, 1)


In [3]:
# Random initializations / seeding
## 2D
tensor_rand_1_1 = torch.randn(1, 1, requires_grad=True)
## 1D
tensor_rand_1 = torch.randn(1, requires_grad=True)
## 3D
tensor_rand_1_1_1 = torch.randn(1, 1, 1, requires_grad=True)

# Now print those:
print(tensor_rand_1_1)
print(tensor_rand_1)
print(tensor_rand_1_1_1)

tensor([[-1.9879]], requires_grad=True)
tensor([-0.0017], requires_grad=True)
tensor([[[0.2667]]], requires_grad=True)


In [4]:
display(pd.DataFrame(data))

,0,1,2,3
0,1.0,2.0,3.0,4.0
1,10.0,20.0,30.0,40.0
2,100.0,200.0,300.0,400.0


In [5]:
# Without grad
tensor_instance_data = torch.tensor(data)
print('------------')
display(tensor_instance_data)
print('------------')
print_tensor_details(tensor_instance_data)


------------


tensor([[  1.,   2.,   3.,   4.],
        [ 10.,  20.,  30.,  40.],
        [100., 200., 300., 400.]])

------------
instance before debug:  tensor([[  1.,   2.,   3.,   4.],
        [ 10.,  20.,  30.,  40.],
        [100., 200., 300., 400.]])
shape:  torch.Size([3, 4])
size():  torch.Size([3, 4])
device:  cpu
dtype:  torch.float32
grad:  None
grad_fn:  None
instance after debug:  tensor([[  1.,   2.,   3.,   4.],
        [ 10.,  20.,  30.,  40.],
        [100., 200., 300., 400.]])


In [6]:
# With Grad
tensor_instance_data_with_grad = torch.tensor(data, requires_grad=True)

# Printing some specific columns create a new set of tensors:
print_tensor_details(tensor_instance_data_with_grad)

instance before debug:  tensor([[  1.,   2.,   3.,   4.],
        [ 10.,  20.,  30.,  40.],
        [100., 200., 300., 400.]], requires_grad=True)
shape:  torch.Size([3, 4])
size():  torch.Size([3, 4])
device:  cpu
dtype:  torch.float32
grad:  None
grad_fn:  None
instance after debug:  tensor([[  1.,   2.,   3.,   4.],
        [ 10.,  20.,  30.,  40.],
        [100., 200., 300., 400.]], requires_grad=True)


In [7]:
print("Col_vect Tensor: ", torch.tensor(col_vect, requires_grad=True, dtype=torch.float))
print("ROw_vect Tensor: ", torch.tensor(row_vect, requires_grad=True, dtype=torch.float))

Col_vect Tensor:  tensor([[2000.],
        [3000.],
        [5000.]], requires_grad=True)
ROw_vect Tensor:  tensor([[2000., 3000., 5000.]], requires_grad=True)


In [8]:
# Operation on Tensors

## dim=1 results into column vect.
col_mean = torch.mean(tensor_instance_data_with_grad, dim=1)
print("Col Vect (row-wise means): ", col_mean)

col_mean_reshaped = torch.mean(tensor_instance_data_with_grad, dim=1).reshape(3, 1)
print("Col Vect (row-wise means) Reshaped: ", col_mean_reshaped)

## dim=0 results into Row vect.
row_mean = torch.mean(tensor_instance_data_with_grad, dim=0)
print("Row Vect (col-wise means): ", row_mean)



## Now, adding col_mean_reshaped to col_vect [3 x 1]
print("\n==============\n\n")
### Throws error as the type do not match for + operator
### col_result = col_vect + col_mean_reshaped
col_result = torch.tensor(col_vect) + col_mean_reshaped
print("Res Column Tensor: ", col_result)

##### ** Incorrect dimensions - but results into another Matrix - so be careful!
#### row_result = torch.tensor(row_vect) + col_mean_reshaped
#### print("Res Row Tensor: ", row_result)
##### ** This one gives error as [a, b, c] cannot be added to [p, q, r, s],
#### even though both are row vects
#### row_result = torch.tensor(row_vect) + row_mean
#### print("Res Row Tensor: ", row_result)


#Simply, for [1 x 3]
row_result = torch.tensor(row_vect, dtype=torch.float32, requires_grad=True)
print("Res Row Tensor: ", row_result)


Col Vect (row-wise means):  tensor([  2.5000,  25.0000, 250.0000], grad_fn=<MeanBackward1>)
Col Vect (row-wise means) Reshaped:  tensor([[  2.5000],
        [ 25.0000],
        [250.0000]], grad_fn=<ViewBackward0>)
Row Vect (col-wise means):  tensor([ 37.,  74., 111., 148.], grad_fn=<MeanBackward1>)



Res Column Tensor:  tensor([[2002.5000],
        [3025.0000],
        [5250.0000]], grad_fn=<AddBackward0>)
Res Row Tensor:  tensor([[2000., 3000., 5000.]], requires_grad=True)


In [9]:
# Pythonic multiplications
## Not supported natively in Py
## print(row_vect @ data)

# Matrix Multiplication
mat_4x1_ = row_result @ tensor_instance_data
print(mat_4x1_)

# Tensor Product
mat_4x4_ = row_result * col_result
print("Mat 4x4 : ", mat_4x4_)



tensor([[ 532000., 1064000., 1596000., 2128000.]], grad_fn=<MmBackward0>)
Mat 4x4 :  tensor([[ 4005000.,  6007500., 10012500.],
        [ 6050000.,  9075000., 15125000.],
        [10500000., 15750000., 26250000.]], grad_fn=<MulBackward0>)


In [10]:
# Dimension-wise Aggregate functions

## Arg-Max - collect the max(per col) ==> result col_max
col_max = torch.argmax(tensor_instance_data_with_grad, dim=1).reshape(3, 1)
row_max = torch.argmax(tensor_instance_data_with_grad, dim=0)
print("Max Col Max: \n", col_max)
print("Max Row Max: ", row_max)

Max Col Max: 
 tensor([[3],
        [3],
        [3]])
Max Row Max:  tensor([2, 2, 2, 2])
